# I3.30R3: Adaptive Authority Causal Isolation

**Three-arm study: V1 vs V3-SHADOW vs V3-AUTH**

Primary question: *Does adaptive hard authority itself improve outcomes when everything else is held constant?*

$$ATE_{authority} = E[U \mid V3\text{-AUTH}] - E[U \mid V3\text{-SHADOW}]$$

185 tasks x 3 arms = 555 trajectories on Colab T4.

**Branch:** `i3.30r3-authority-isolation`
**Commit:** `5da77aa`

**Treatment purity invariant:** V3-SHADOW and V3-AUTH execute identical code up to the final override:
```python
# Everything above this line must be identical.
if arm == V3_HARD and decision.would_force:
    action = decision.forced_action
else:
    action = llm_action
```

## 1. Environment Setup

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Install llama-cpp-python with CUDA support
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python==0.3.35 --no-cache-dir

In [ ]:
# Install other dependencies
!pip install scikit-learn numpy --quiet

## 2. Clone Repository and Verify Commit

In [ ]:
import subprocess

REPO_URL = "https://github.com/dawsonblock/Daph-ex-research-gate-c2-beir-retrieval.git"
BRANCH = "i3.30r3-authority-isolation"
EXPECTED_COMMIT = "5da77aa9e4f905f9d8481cfeb79b20d057edd1ef"

# Clone if not already present
import os
if not os.path.exists('/content/daph'):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, '/content/daph'], check=True)

os.chdir('/content/daph')

# Verify commit
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print(f'Current commit: {commit[:12]}')
print(f'Expected commit: {EXPECTED_COMMIT[:12]}')
assert commit == EXPECTED_COMMIT, f'Commit mismatch! Got {commit}, expected {EXPECTED_COMMIT}'
print('Commit verified.')

# Show branch and status
branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip()
print(f'Branch: {branch}')
status = subprocess.run(['git', 'status', '--short'], capture_output=True, text=True)
if status.stdout.strip():
    print(f'WARNING: Working tree is dirty:\n{status.stdout}')
else:
    print('Working tree is clean.')

## 3. Download GGUF Model

In [ ]:
import hashlib, os

GGUF_PATH = '/content/Qwen2.5-7B-Instruct-Q4_K_M.gguf'

# Download from official Qwen HF repo
if not os.path.exists(GGUF_PATH):
    !wget -q https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/resolve/main/qwen2.5-7b-instruct-q4_k_m.gguf -O {GGUF_PATH}

# Compute SHA
with open(GGUF_PATH, 'rb') as f:
    gguf_sha = hashlib.sha256(f.read()).hexdigest()
print(f'GGUF SHA256: {gguf_sha}')
print(f'GGUF size: {os.path.getsize(GGUF_PATH) / 1e9:.2f} GB')

# Note: This SHA may differ from the local Metal GGUF.
# The I3.30R2 diagnostic noted this as a development concern.
# For I3.30R3, we record the exact Colab SHA in the manifest.
# This is a backend replication, not byte-identical model equivalence.

## 4. Run Treatment-Purity Tests

In [ ]:
%cd /content/daph
!PYTHONPATH=. python3 -m pytest tests/unit/test_i3_30r3_authority_isolation.py -v --timeout=30

In [ ]:
# Run existing authority and topology tests to verify no regressions
!PYTHONPATH=. python3 -m pytest tests/unit/test_authority_v2.py tests/unit/test_authority_v3.py tests/unit/test_epistemic_topology.py -v --timeout=30

## 5. Freeze Manifest (Pre-Run Verification)

In [ ]:
%cd /content/daph
!PYTHONPATH=. python3 scripts/run_i3_30r3_authority_isolation.py \
    --gguf-path /content/Qwen2.5-7B-Instruct-Q4_K_M.gguf \
    --output-dir experiments/i3_30r3/live \
    --freeze-manifest-only

In [ ]:
# Verify manifest SHAs match preregistration
import json

with open('experiments/i3_30r3/live/frozen_manifest.json') as f:
    manifest = json.load(f)
with open('experiments/i3_30r3/I3_30R3_PREREGISTRATION.json') as f:
    prereg = json.load(f)

mismatches = []
for key, expected in prereg['frozen_artifacts'].items():
    actual = manifest.get(key, 'MISSING')
    if actual != expected:
        mismatches.append(f'  {key}: expected {expected[:16]}..., got {actual[:16]}...')

if mismatches:
    print('*** MANIFEST MISMATCHES ***')
    for m in mismatches:
        print(m)
    print('ABORT: Frozen artifacts do not match preregistration.')
else:
    print('All frozen artifact SHAs match preregistration.')
    print(f'Task count: {manifest["task_count"]}')
    print(f'Trajectory count: {manifest["trajectory_count"]}')
    print(f'Arms: {manifest["arms"]}')
    print(f'Gates: {len(manifest["gates"])}')

## 6. Run D5 State-Truth Audit (Quick Verification)

In [ ]:
%cd /content/daph
!PYTHONPATH=. python3 scripts/audit_d5_state_truth.py 2>&1 | head -20

## 7. Run Three-Arm Study (555 Trajectories)

This is the main experiment. Expected time: ~35-40 minutes on T4.

185 tasks x 3 arms (V1, V3-SHADOW, V3-AUTH) = 555 trajectories.

In [ ]:
%cd /content/daph
import time
start = time.time()

!PYTHONPATH=. python3 scripts/run_i3_30r3_authority_isolation.py \
    --gguf-path /content/Qwen2.5-7B-Instruct-Q4_K_M.gguf \
    --output-dir experiments/i3_30r3/live \
    --resume

elapsed = time.time() - start
print(f'\nTotal elapsed: {elapsed:.1f}s ({elapsed/60:.1f} min)')

## 8. Evaluate Results

In [ ]:
%cd /content/daph
!PYTHONPATH=. python3 scripts/evaluate_i3_30r3_authority_isolation.py \
    --input-dir experiments/i3_30r3/live \
    --output-dir experiments/i3_30r3/analysis

## 9. Quick Results Summary

In [ ]:
import json
from pathlib import Path

# Load analysis
with open('experiments/i3_30r3/analysis/authority_analysis.json') as f:
    analysis = json.load(f)

print('=' * 60)
print('PRIMARY COMPARISON: V3-AUTH vs V3-SHADOW')
print('=' * 60)
primary = analysis['primary_comparison']
ate = primary['ate_authority']
print(f'  ATE_authority = {ate["mean"]:.4f}')
print(f'  95% CI: [{ate["lower"]:.4f}, {ate["upper"]:.4f}]')
print(f'  n = {ate["n"]}')
sd = primary['success_delta']
print(f'  Rescues: {sd["rescues"]}')
print(f'  Breaks: {sd["breaks"]}')
print(f'  Both success: {sd["both_success"]}')
print(f'  Both fail: {sd["both_fail"]}')

print()
print('=' * 60)
print('SECONDARY COMPARISON: V3-SHADOW vs V1')
print('=' * 60)
secondary = analysis['secondary_comparison']
du = secondary['delta_utility']
print(f'  Delta_U(SHADOW-V1) = {du["mean"]:.4f}')
print(f'  95% CI: [{du["lower"]:.4f}, {du["upper"]:.4f}]')
sd2 = secondary['success_delta']
print(f'  Rescues: {sd2["rescues"]}')
print(f'  Breaks: {sd2["breaks"]}')

print()
print('=' * 60)
print('AGGREGATE')
print('=' * 60)
agg = analysis['aggregate']
print(f'  V1:      {agg["v1_success"]}/{agg["v1_total"]} = {agg["v1_success"]/agg["v1_total"]:.2%}')
print(f'  SHADOW:  {agg["shadow_success"]}/{agg["shadow_total"]} = {agg["shadow_success"]/agg["shadow_total"]:.2%}')
print(f'  AUTH:    {agg["hard_success"]}/{agg["hard_total"]} = {agg["hard_success"]/agg["hard_total"]:.2%}')

print()
print('=' * 60)
print('AUTHORITY EVENT CLASSIFICATION')
print('=' * 60)
for effect, count in sorted(analysis['effect_classification'].items()):
    print(f'  {effect}: {count}')

print()
print('=' * 60)
print('AUTHORITY RATES')
print('=' * 60)
rates = analysis['authority_rates']
print(f'  Certificate coverage: {rates["certificate_coverage"]:.4f}')
print(f'  Force rate: {rates["force_rate"]:.4f}')
print(f'  Effective intervention rate: {rates["effective_intervention_rate"]:.4f}')
print(f'  Certificate positive: {rates["certificate_positive_count"]}')
print(f'  Force applied: {rates["force_applied_count"]}')
print(f'  Effective interventions: {rates["effective_intervention_count"]}')

In [ ]:
# Load gate evaluation
with open('experiments/i3_30r3/analysis/gate_evaluation.json') as f:
    gates = json.load(f)

print('=' * 60)
print('GATE EVALUATION')
print('=' * 60)
for gid, gate in sorted(gates['gates'].items()):
    status = gate['result']
    value = gate.get('value', '')
    print(f'  {gid} {gate["name"]:<30} {status:<8} {value}')
print(f'\n  Passed: {gates["passed"]}, Failed: {gates["failed"]}, Pending: {gates["pending"]}')

In [ ]:
# Stratum breakdown
strata = analysis['strata']
print('=' * 60)
print('STRATUM BREAKDOWN')
print('=' * 60)
print(f'  {"Stratum":<30} {"V1":>10} {"SHADOW":>10} {"AUTH":>10}')
for stratum in sorted(set(strata['v1']) | set(strata['v3_shadow']) | set(strata['v3_hard'])):
    v1_s = strata['v1'].get(stratum, {}).get('success_rate', 0)
    sh_s = strata['v3_shadow'].get(stratum, {}).get('success_rate', 0)
    hd_s = strata['v3_hard'].get(stratum, {}).get('success_rate', 0)
    print(f'  {stratum:<30} {v1_s:>10.2%} {sh_s:>10.2%} {hd_s:>10.2%}')

## 10. Package Results for Download

In [ ]:
import shutil

# Package all results
output_zip = '/content/i3_30r3_results.zip'
shutil.make_archive('/content/i3_30r3_results', 'zip', 'experiments/i3_30r3')
print(f'Results packaged to {output_zip}')
print(f'Size: {os.path.getsize(output_zip) / 1e6:.1f} MB')
print()
print('Files included:')
for root, dirs, files in os.walk('experiments/i3_30r3'):
    for f in files:
        rel = os.path.relpath(os.path.join(root, f), 'experiments/i3_30r3')
        size = os.path.getsize(os.path.join(root, f))
        print(f'  {rel:<50} {size:>10,} bytes')

## 11. Record Run Provenance

Record the exact runtime environment for provenance.

In [ ]:
import json, subprocess, platform, time

# Record provenance
provenance = {
    'experiment': 'I3.30R3',
    'run_timestamp': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()),
    'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip(),
    'branch': subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip(),
    'platform': platform.platform(),
    'python_version': platform.python_version(),
    'gguf_sha256': gguf_sha,
    'gguf_path': GGUF_PATH,
    'gpu': subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip(),
}

# Try to get llama-cpp-python version
try:
    import llama_cpp
    provenance['llama_cpp_version'] = llama_cpp.__version__
except:
    provenance['llama_cpp_version'] = 'unknown'

with open('experiments/i3_30r3/live/run_provenance.json', 'w') as f:
    json.dump(provenance, f, indent=2)

print('Run provenance:')
for k, v in provenance.items():
    print(f'  {k}: {v}')

## 12. Re-package with Provenance and Download

In [ ]:
# Re-package with provenance included
shutil.make_archive('/content/i3_30r3_results', 'zip', 'experiments/i3_30r3')
print(f'Final results package: /content/i3_30r3_results.zip')
print(f'Size: {os.path.getsize("/content/i3_30r3_results.zip") / 1e6:.1f} MB')
print()
print('Download this file from the Colab file browser (left sidebar -> Files tab).')
print('Or run:')
print('  from google.colab import files')
print('  files.download("/content/i3_30r3_results.zip")')

In [ ]:
# Auto-download (may not work in all browsers)
try:
    from google.colab import files
    files.download('/content/i3_30r3_results.zip')
except:
    print('Auto-download not available. Please download manually from the Files tab.')